# 06 · LiDAR 点云与 BEV Occupancy

从二维图像进入自动驾驶感知，必须理解点云坐标、体素化和 BEV 表示。这里生成三个带高度的合成物体，把点云投影成 BEV occupancy，再注入点 dropout、测量噪声和分辨率变化。

学习目标：

- 区分 sensor frame、ego frame 和 BEV grid index。
- 理解 voxel / rasterization 如何把不规则点云变成神经网络可处理的张量。
- 观察稀疏性、噪声和分辨率对 occupancy IoU 的影响。
- 认识为什么真实 3D detector 还需要 box parameterization、NMS 和 tracking。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
from ipywidgets import interact, FloatSlider, IntSlider

plt.rcParams['figure.figsize'] = (9, 4.5)
plt.rcParams['axes.grid'] = True
BOUNDS = (-10.0, 35.0, -12.0, 12.0)
rng = np.random.default_rng(4)


In [ ]:
def make_scene(dropout=0.0, noise=0.05, seed=4):
    local = np.random.default_rng(seed)
    objects = [
        {'center': np.array([8.0, -3.0, 1.0]), 'size': np.array([4.0, 1.8, 1.5])},
        {'center': np.array([18.0, 3.5, 1.2]), 'size': np.array([3.0, 1.6, 1.4])},
        {'center': np.array([27.0, -1.0, 1.0]), 'size': np.array([5.0, 2.0, 1.6])},
    ]
    clouds = []
    for obj in objects:
        count = 700
        pts = local.uniform(-0.5, 0.5, size=(count, 3)) * obj['size'] + obj['center']
        clouds.append(pts)
    ground_count = 1600
    ground = np.c_[
        local.uniform(BOUNDS[0], BOUNDS[1], ground_count),
        local.uniform(BOUNDS[2], BOUNDS[3], ground_count),
        local.normal(0.0, 0.03, ground_count),
    ]
    points = np.vstack([ground] + clouds)
    keep = local.random(len(points)) > dropout
    points = points[keep] + local.normal(0.0, noise, size=(keep.sum(), 3))
    return points, objects

def rasterize(points, resolution=0.5):
    width = int(np.ceil((BOUNDS[1] - BOUNDS[0]) / resolution))
    height = int(np.ceil((BOUNDS[3] - BOUNDS[2]) / resolution))
    grid = np.zeros((height, width), dtype=np.uint8)
    ix = ((points[:, 0] - BOUNDS[0]) / resolution).astype(int)
    iy = ((points[:, 1] - BOUNDS[2]) / resolution).astype(int)
    valid = (ix >= 0) & (ix < width) & (iy >= 0) & (iy < height) & (points[:, 2] > 0.25)
    grid[iy[valid], ix[valid]] = 1
    return grid

def object_mask(objects, resolution=0.5):
    width = int(np.ceil((BOUNDS[1] - BOUNDS[0]) / resolution))
    height = int(np.ceil((BOUNDS[3] - BOUNDS[2]) / resolution))
    mask = np.zeros((height, width), dtype=np.uint8)
    for obj in objects:
        center, size = obj['center'], obj['size']
        x0 = int((center[0] - size[0] / 2 - BOUNDS[0]) / resolution)
        x1 = int((center[0] + size[0] / 2 - BOUNDS[0]) / resolution)
        y0 = int((center[1] - size[1] / 2 - BOUNDS[2]) / resolution)
        y1 = int((center[1] + size[1] / 2 - BOUNDS[2]) / resolution)
        mask[max(y0, 0):min(y1 + 1, height), max(x0, 0):min(x1 + 1, width)] = 1
    return mask

def iou(a, b):
    intersection = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return intersection / max(union, 1)

points, objects = make_scene()
print(f'points: {len(points)}')
print('object centers:', [obj['center'].round(1).tolist() for obj in objects])


In [ ]:
def show_bev(dropout=0.15, noise=0.05, resolution=0.5):
    points, objects = make_scene(dropout=dropout, noise=noise)
    observed = rasterize(points, resolution)
    truth = object_mask(objects, resolution)
    cleaned = ndimage.binary_opening(observed, structure=np.ones((2, 2)))
    score_observed = iou(observed, truth)
    score_cleaned = iou(cleaned, truth)
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].scatter(points[:, 0], points[:, 1], c=points[:, 2], s=1, cmap='viridis')
    ax[0].set_title(f'point cloud: N={len(points)}')
    ax[0].set_xlabel('x / m')
    ax[0].set_ylabel('y / m')
    ax[1].imshow(truth, origin='lower', extent=BOUNDS, aspect='auto', cmap='Greens')
    ax[1].set_title('ground-truth object mask')
    ax[2].imshow(cleaned, origin='lower', extent=BOUNDS, aspect='auto', cmap='magma')
    ax[2].set_title(f'BEV occupancy IoU={score_cleaned:.3f} (raw={score_observed:.3f})')
    for axis in ax:
        axis.set_xlabel('x / m')
    plt.tight_layout()
    plt.show()

interact(
    show_bev,
    dropout=FloatSlider(min=0.0, max=0.85, step=0.05, value=0.15, description='dropout'),
    noise=FloatSlider(min=0.0, max=0.35, step=0.025, value=0.05, description='xyz noise'),
    resolution=FloatSlider(min=0.25, max=1.5, step=0.25, value=0.5, description='grid m'),
);


### 练习：从 occupancy 走向 3D detection

- 比较 0.25 m、0.5 m、1.0 m 分辨率下的 IoU 和点数。
- 给三个物体加入不同反射率或不同点密度，设计一个 density-aware score。
- 用 connected components 得到候选框，并报告 box IoU，而不仅是 occupancy IoU。
- 解释为什么 BEV rasterization 的坐标方向、原点和分辨率必须写进数据契约。


In [ ]:
resolution_values = [0.25, 0.5, 1.0]
scores = []
for resolution in resolution_values:
    pts, objs = make_scene(dropout=0.25, noise=0.08)
    scores.append(iou(rasterize(pts, resolution), object_mask(objs, resolution)))
plt.bar([str(v) for v in resolution_values], scores)
plt.xlabel('BEV resolution / m')
plt.ylabel('occupancy IoU')
plt.title('resolution ablation under the same corruption')
plt.show()
print(dict(zip(resolution_values, np.round(scores, 3))))


## 完成标准

保留一张点云与 BEV 对照图、一张分辨率或 dropout ablation 图，并回答：

1. 你的 BEV index 如何从 ego-frame 坐标得到？
2. 该 toy occupancy 与真实 CenterPoint、BEVFusion 或 Occupancy 网络之间差在哪里？
3. 哪些误差来自传感器，哪些误差来自 rasterizer，哪些误差来自模型？
